# 🚀 HappyGen Studio - Google Colab 16GB Cloud GPU Server
**Free Tesla T4 / A100 GPU • Full API: txt2img, img2img, interrogate, upscale, facefix**

### Instructions:
1. Go to **Runtime -> Change runtime type -> Select T4 GPU**.
2. Run all cells below.
3. Copy the public **Cloudflare Tunnel URL** (`https://xxxx.trycloudflare.com`).
4. In HappyGen Web App, click the **Settings / Backend** icon in the header, paste the URL, and click **Test Connection**!

In [1]:
# Cell 1: Install Dependencies
!pip install -q diffusers transformers accelerate safetensors sentencepiece protobuf fastapi uvicorn pydantic pycloudflared nest_asyncio python-multipart peft open_clip_torch compel
!rm -rf BasicSR
!git clone https://github.com/xinntao/BasicSR.git
!cd BasicSR && sed -i 's/return locals().*/return "1.4.2"/g' setup.py && pip install -q .
!python -c "import sys; sys.path.append('/usr/local/lib/python3.10/dist-packages'); import basicsr.data.degradations as d; open(d.__file__, 'w').write(open(d.__file__).read().replace('from torchvision.transforms.functional_tensor import rgb_to_grayscale', 'from torchvision.transforms.functional import rgb_to_grayscale'))"
!pip install -q facexlib realesrgan gfpgan
















In [2]:
# Cell 2: Download Models & SDXL Lightning Accelerator
import os
os.makedirs("/content/Models", exist_ok=True)
os.makedirs("/content/LoRAs", exist_ok=True)
os.makedirs("/content/Embeddings", exist_ok=True)

# Define the path for the desired Civitai base model
BASE_MODEL_PATH = "/content/Models/crucibleRINGPonyxl_v28.safetensors"
LIGHTNING_PATH = "/content/LoRAs/sdxl_lightning_4step_lora.safetensors"

# Import userdata for secrets
from google.colab import userdata

try:
    CIVITAI_API_KEY = userdata.get('CIVITAI_API_KEY')
except Exception:
    CIVITAI_API_KEY = None

def civitai_download_url(model_version_id):
    base_url = f"https://civitai.com/api/download/models/{model_version_id}"
    if CIVITAI_API_KEY:
        return f"{base_url}?token={CIVITAI_API_KEY}"
    return base_url

# Clean up corrupted files
if os.path.exists(BASE_MODEL_PATH) and os.path.getsize(BASE_MODEL_PATH) < 1024 * 1024:
    print(f"⚠️ Found corrupted or incomplete file at {BASE_MODEL_PATH}. Deleting and re-downloading...")
    os.remove(BASE_MODEL_PATH)

if not os.path.exists(BASE_MODEL_PATH):
    print("📥 Downloading CrucibleRING PonyXL v28 (~6.6GB) from Civitai...")
    !wget -c "{civitai_download_url('1979291')}" -O {BASE_MODEL_PATH}
    if os.path.exists(BASE_MODEL_PATH) and os.path.getsize(BASE_MODEL_PATH) < 1024 * 1024:
        print(f"⚠️ Download failed. It might be corrupted. Deleting file.")
        os.remove(BASE_MODEL_PATH)
    else:
        print(f"✅ Base model downloaded successfully.")

if not os.path.exists(LIGHTNING_PATH):
    print("⚡ Downloading SDXL Lightning 4-Step LoRA...")
    !wget -c "https://huggingface.co/ByteDance/SDXL-Lightning/resolve/main/sdxl_lightning_4step_lora.safetensors" -O {LIGHTNING_PATH}

print("✅ Storage ready! Models directory is prepared.")
















In [3]:
# Cell 3: Prepare GPU (model loads on first request)
import torch
from diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline, EulerAncestralDiscreteScheduler

print(f"🚀 GPU ready: {torch.cuda.get_device_name(0)} with {torch.cuda.mem_get_info()[0] / (1024**3):.1f} GB free VRAM")
CURRENT_BASE_MODEL_FILE = None
pipe = None
pipe_img2img = None

print("✅ Server initialized. Model will load on first generation request.")
















In [4]:
# Cell 4: Launch FastAPI Server & Cloudflare Public Tunnel
import io, base64, time, json, threading, nest_asyncio, os, uuid
import numpy as np
from PIL import Image
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional, Union
import uvicorn
from pycloudflared import try_cloudflare
import requests
import gc
import subprocess

nest_asyncio.apply()
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ============================================================================
# Lazy-loaded utility models (loaded on first use, unloaded after to save VRAM)
# ============================================================================
_clip_model = None
_clip_preprocess = None
_upscaler = None
_face_restorer = None

tasks = {}
def _bg_runner(task_id, func, *args, **kwargs):
    try:
        res = func(*args, **kwargs)
        tasks[task_id]['status'] = 'completed'
        tasks[task_id]['result'] = res
    except Exception as e:
        tasks[task_id]['status'] = 'failed'
        tasks[task_id]['error'] = str(e)

@app.get("/async/status/{task_id}")
def async_status(task_id: str):
    if task_id not in tasks: return {"status": "not_found"}
    return tasks[task_id]

_tagger_session = None
_tagger_tags = None

def _load_tagger():
    global _tagger_session, _tagger_tags
    if _tagger_session is not None: return
    import subprocess
    import csv
    print("📦 Installing WD14 Tagger dependencies...")
    subprocess.check_call(["pip", "install", "-q", "onnxruntime-gpu", "huggingface_hub"])
    from huggingface_hub import hf_hub_download
    import onnxruntime as ort
    repo_id = "SmilingWolf/wd-v1-4-moat-tagger-v2"
    print("🔍 Downloading WD14 Tagger Model...")
    model_path = hf_hub_download(repo_id, "model.onnx")
    tags_path = hf_hub_download(repo_id, "selected_tags.csv")
    with open(tags_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader)
        _tagger_tags = [row[1] for row in reader]
    _tagger_session = ort.InferenceSession(model_path, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])

def _unload_tagger():
    global _tagger_session, _tagger_tags
    if _tagger_session is not None:
        del _tagger_session
        _tagger_session = None
        _tagger_tags = None
        gc.collect()
        torch.cuda.empty_cache()

def _load_upscaler():
    global _upscaler
    if _upscaler is not None: return
    from realesrgan import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet
    print("🔎 Loading Real-ESRGAN upscaler...")
    model_path = "/content/Models/RealESRGAN_x4plus.pth"
    if not os.path.exists(model_path):
        !wget -q -c "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth" -O {model_path}
    rrdb_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    _upscaler = RealESRGANer(scale=4, model_path=model_path, model=rrdb_model, tile=256, tile_pad=10, pre_pad=0, half=True, gpu_id=0)

def _unload_upscaler():
    global _upscaler
    if _upscaler is not None:
        del _upscaler
        _upscaler = None
        gc.collect()
        torch.cuda.empty_cache()

def _load_face_restorer():
    global _face_restorer
    if _face_restorer is not None: return
    from gfpgan import GFPGANer
    print("👤 Loading GFPGAN face restorer...")
    model_path = "/content/Models/GFPGANv1.4.pth"
    if not os.path.exists(model_path):
        !wget -q -c "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth" -O {model_path}
    _face_restorer = GFPGANer(model_path=model_path, upscale=1, arch='clean', channel_multiplier=2, bg_upsampler=None)

def _unload_face_restorer():
    global _face_restorer
    if _face_restorer is not None:
        del _face_restorer
        _face_restorer = None
        gc.collect()
        torch.cuda.empty_cache()

def _decode_base64_image(b64_string):
    if "," in b64_string:
        b64_string = b64_string.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_string))).convert("RGB")

def _encode_image_to_base64(pil_image):
    buffered = io.BytesIO()
    pil_image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

class Txt2ImgRequest(BaseModel):
    prompt: str
    negative_prompt: Optional[str] = "score_4, score_5, score_6, bad hands, blurry, low quality"
    steps: Optional[int] = 20
    cfg_scale: Optional[float] = 6.5
    width: Optional[int] = 832
    height: Optional[int] = 1216
    seed: Optional[int] = -1
    sampler_name: Optional[str] = "Euler a"
    base_model: Optional[Union[dict, str]] = "crucibleRINGPonyxl_v28.safetensors"
    loras: Optional[List[Union[dict, str]]] = []
    embeddings: Optional[List[Union[dict, str]]] = []
    civitai_api_key: Optional[str] = ""

class Img2ImgRequest(BaseModel):
    prompt: str
    negative_prompt: Optional[str] = "score_4, score_5, score_6, bad hands, blurry, low quality"
    init_images: List[str]
    denoising_strength: Optional[float] = 0.5
    steps: Optional[int] = 20
    cfg_scale: Optional[float] = 6.5
    width: Optional[int] = 832
    height: Optional[int] = 1216
    seed: Optional[int] = -1
    sampler_name: Optional[str] = "Euler a"
    mask: Optional[str] = None
    base_model: Optional[Union[dict, str]] = None
    loras: Optional[List[Union[dict, str]]] = []
    embeddings: Optional[List[Union[dict, str]]] = []
    civitai_api_key: Optional[str] = ""

class InterrogateRequest(BaseModel):
    image: str
    model: Optional[str] = "clip"

class UpscaleRequest(BaseModel):
    image: str
    upscaling_resize: Optional[int] = 2

class FaceFixRequest(BaseModel):
    image: str
    prompt: Optional[str] = ""

def download_civitai_model(download_url, dest_path, api_key):
    if os.path.exists(dest_path): return True
    print(f"Downloading missing model to {os.path.basename(dest_path)}...")
    active_key = api_key if api_key else CIVITAI_API_KEY
    headers = {}
    if active_key: headers["Authorization"] = f"Bearer {active_key}"
    
    response = requests.get(download_url, headers=headers, stream=True, allow_redirects=True)
    if response.status_code != 200:
        raise RuntimeError(f"Download failed with HTTP {response.status_code}. Make sure your Civitai API key is valid if this is a gated model.")
    
    content_type = response.headers.get("Content-Type", "")
    if "text/html" in content_type:
        raise RuntimeError(f"Civitai API returned an HTML webpage instead of a model file. The download URL is likely incorrect or requires an API key. URL: {download_url}")
        
    with open(dest_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            
    if os.path.getsize(dest_path) < 1024:
        os.remove(dest_path)
        raise RuntimeError(f"Downloaded file is suspiciously small (<1KB). The API key might be invalid or the URL is broken.")
        
    return True

def _apply_loras(target_pipe, loras, api_key):
    loaded_adapters = []
    loaded_weights = []
    if not loras or not target_pipe: return loaded_adapters
    for item in loras:
        name = item if isinstance(item, str) else item.get("fileName") or item.get("name")
        weight = 0.85 if isinstance(item, str) else float(item.get("weight", 0.85))
        if not name: continue
        lora_file = name if name.endswith(".safetensors") else f"{name}.safetensors"
        lora_path = os.path.join("/content/LoRAs", lora_file)
        lora_url = item.get("downloadUrl") if isinstance(item, dict) else None
        if not os.path.exists(lora_path) and lora_url:
            download_civitai_model(lora_url, lora_path, api_key)
        if os.path.exists(lora_path) and lora_file != CURRENT_BASE_MODEL_FILE and lora_file != os.path.basename(LIGHTNING_PATH):
            try:
                adapter_id = f"lora_{len(loaded_adapters)}"
                target_pipe.load_lora_weights("/content/LoRAs", weight_name=lora_file, adapter_name=adapter_id)
                loaded_weights.append(weight)
                loaded_adapters.append(adapter_id)
            except Exception as e:
                print(f"LoRA load note: {e}")
    if loaded_adapters:
        try:
            target_pipe.set_adapters(loaded_adapters, adapter_weights=loaded_weights)
        except Exception as e:
            print(f"Failed to set adapters: {e}")
            try: target_pipe.delete_adapters(loaded_adapters)
            except: pass
            loaded_adapters = []
    return loaded_adapters

def _apply_sampler(target_pipe, sampler_name):
    if not sampler_name or "Flux" in str(type(target_pipe)): return
    from diffusers import (EulerAncestralDiscreteScheduler, EulerDiscreteScheduler, DPMSolverMultistepScheduler, DPMSolverSinglestepScheduler, UniPCMultistepScheduler, DDIMScheduler, LMSDiscreteScheduler, PNDMScheduler)
    cfg = target_pipe.scheduler.config
    if sampler_name == "Euler a": target_pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(cfg)
    elif sampler_name == "DPM++ 2M Karras": target_pipe.scheduler = DPMSolverMultistepScheduler.from_config(cfg, use_karras_sigmas=True)
    elif sampler_name == "DPM++ SDE Karras": target_pipe.scheduler = DPMSolverSinglestepScheduler.from_config(cfg, use_karras_sigmas=True)
    elif sampler_name == "Euler": target_pipe.scheduler = EulerDiscreteScheduler.from_config(cfg)
    elif sampler_name == "UniPC": target_pipe.scheduler = UniPCMultistepScheduler.from_config(cfg)
    elif sampler_name == "DDIM": target_pipe.scheduler = DDIMScheduler.from_config(cfg)
    elif sampler_name == "LMS Karras": target_pipe.scheduler = LMSDiscreteScheduler.from_config(cfg, use_karras_sigmas=True)
    elif sampler_name == "PNDM": target_pipe.scheduler = PNDMScheduler.from_config(cfg)


def _do_txt2img(req: Txt2ImgRequest):
    _switch_model_if_needed(req.base_model, req.civitai_api_key)
    seed = req.seed if (req.seed is not None and req.seed >= 0) else int(torch.randint(0, 2**32, (1,)).item())
    generator = torch.Generator("cuda").manual_seed(seed)
    _load_embeddings(pipe, req.embeddings, req.civitai_api_key)
    loaded_adapters = _apply_loras(pipe, req.loras, req.civitai_api_key)
    _apply_sampler(pipe, getattr(req, "sampler_name", "Euler a"))
    prompt_str = req.prompt if "score_" in req.prompt else f"score_9, score_8_up, score_7_up, source_anime, {req.prompt}"

    image = None
    try:
        from compel import Compel, ReturnedEmbeddingsType
        if "Flux" not in str(type(pipe)):
            if "StableDiffusionXL" in str(type(pipe)):
                compel_proc = Compel(tokenizer=[pipe.tokenizer, pipe.tokenizer_2], text_encoder=[pipe.text_encoder, pipe.text_encoder_2], returned_embeddings_type=ReturnedEmbeddingsType.PENULTIMATE_HIDDEN_STATES_NON_NORMALIZED, requires_pooled=[False, True], truncate_long_prompts=False, device=pipe.device)
                cond, pooled = compel_proc(prompt_str)
                neg_cond, neg_pooled = compel_proc(req.negative_prompt or "")
                try:
                    try:
                cond, neg_cond = compel_proc.pad_conditioning_tensors_to_same_length([cond, neg_cond])
            except AttributeError:
                import torch
                empty_cond, _ = compel_proc("")
                while cond.shape[1] > neg_cond.shape[1]:
                    neg_cond = torch.cat([neg_cond, empty_cond], dim=1)
                while neg_cond.shape[1] > cond.shape[1]:
                    cond = torch.cat([cond, empty_cond], dim=1)
                except AttributeError:
                    import torch
                    empty_cond, _ = compel_proc("")
                    while cond.shape[1] > neg_cond.shape[1]:
                        neg_cond = torch.cat([neg_cond, empty_cond], dim=1)
                    while neg_cond.shape[1] > cond.shape[1]:
                        cond = torch.cat([cond, empty_cond], dim=1)
                
                # Robustly extract pooled embeddings whether they are list, tuple, or single tensor
                def extract_pool(p):
                    if hasattr(p, 'shape'): return p
                    if isinstance(p, (list, tuple)):
                        valid = [x for x in p if x is not None and hasattr(x, 'shape')]
                        if valid: return valid[0]
                    return None
                    
                pooled_ext = extract_pool(pooled)
                neg_pooled_ext = extract_pool(neg_pooled)
                
                if pooled_ext is None or neg_pooled_ext is None:
                    # Fallback to diffusers encode_prompt for pooled if compel didn't generate them
                    _, _, p_emb, np_emb = pipe.encode_prompt(prompt=prompt_str, prompt_2=prompt_str, device=pipe.device, num_images_per_prompt=1, do_classifier_free_guidance=True, negative_prompt=req.negative_prompt or "", negative_prompt_2=req.negative_prompt or "")
                    pooled_ext = p_emb if pooled_ext is None else pooled_ext
                    neg_pooled_ext = np_emb if neg_pooled_ext is None else neg_pooled_ext
                    
                kwargs = {"prompt_embeds": cond, "pooled_prompt_embeds": pooled_ext, "negative_prompt_embeds": neg_cond, "negative_pooled_prompt_embeds": neg_pooled_ext}
            else:
                compel_proc = Compel(tokenizer=pipe.tokenizer, text_encoder=pipe.text_encoder, truncate_long_prompts=False, device=pipe.device)
                cond = compel_proc(prompt_str)
                neg_cond = compel_proc(req.negative_prompt or "")
                try:
                    try:
                cond, neg_cond = compel_proc.pad_conditioning_tensors_to_same_length([cond, neg_cond])
            except AttributeError:
                import torch
                empty_cond, _ = compel_proc("")
                while cond.shape[1] > neg_cond.shape[1]:
                    neg_cond = torch.cat([neg_cond, empty_cond], dim=1)
                while neg_cond.shape[1] > cond.shape[1]:
                    cond = torch.cat([cond, empty_cond], dim=1)
                except AttributeError:
                    import torch
                    empty_cond, _ = compel_proc("")
                    while cond.shape[1] > neg_cond.shape[1]:
                        neg_cond = torch.cat([neg_cond, empty_cond], dim=1)
                    while neg_cond.shape[1] > cond.shape[1]:
                        cond = torch.cat([cond, empty_cond], dim=1)
                kwargs = {"prompt_embeds": cond, "negative_prompt_embeds": neg_cond}
        else:
            kwargs = {"prompt": prompt_str}

        try:
            with torch.inference_mode():
                image = pipe(**kwargs, num_inference_steps=req.steps, guidance_scale=req.cfg_scale, width=req.width, height=req.height, generator=generator).images[0]
        except Exception as pipe_err:
            pe_type = type(kwargs.get('pooled_prompt_embeds'))
            pe_val = kwargs.get('pooled_prompt_embeds') is None
            raise RuntimeError(f"PIPE CRASH! kwargs keys: {list(kwargs.keys())}. pooled type: {pe_type}. Is it None? {pe_val}. Original error: {pipe_err}")

    except Exception as e:
        import traceback
        import textwrap
        from PIL import Image, ImageDraw
        traceback.print_exc()
        err_msg = traceback.format_exc()
        print(f"Compel processing or generation failed: {e}. Returning error as image.")
        img = Image.new('RGB', (req.width, req.height), color = (30, 30, 30))
        d = ImageDraw.Draw(img)
        y_text = 20
        for line in err_msg.split('\n'):
            for wrap_line in textwrap.wrap(line, width=70):
                d.text((20, y_text), wrap_line, fill=(255, 100, 100))
                y_text += 20
        image = img

    if loaded_adapters:
        try: pipe.delete_adapters(loaded_adapters)
        except: pass

    return {
        "images": [_encode_image_to_base64(image)],
        "source": f"Google Colab Cloud GPU ({torch.cuda.get_device_name(0)})"
    }

@app.post("/sdapi/v1/txt2img")
def txt2img(req: Txt2ImgRequest):
    task_id = str(uuid.uuid4())
    tasks[task_id] = {"status": "processing"}
    threading.Thread(target=_bg_runner, args=(task_id, _do_txt2img, req)).start()
    return {"task_id": task_id}

def _do_img2img(req: Img2ImgRequest):
    _switch_model_if_needed(req.base_model, req.civitai_api_key)
    if not pipe_img2img:
        raise HTTPException(status_code=400, detail="img2img is not supported on this model architecture yet.")
    seed = req.seed if (req.seed is not None and req.seed >= 0) else int(torch.randint(0, 2**32, (1,)).item())
    generator = torch.Generator("cuda").manual_seed(seed)
    init_image = _decode_base64_image(req.init_images[0]).resize((req.width, req.height), Image.LANCZOS)
    _load_embeddings(pipe_img2img, req.embeddings, req.civitai_api_key)
    loaded_adapters = _apply_loras(pipe_img2img, req.loras, req.civitai_api_key)
    _apply_sampler(pipe_img2img, getattr(req, "sampler_name", "Euler a"))
    prompt_str = req.prompt if "score_" in req.prompt else f"score_9, score_8_up, score_7_up, source_anime, {req.prompt}"

    image = None
    try:
        from compel import Compel, ReturnedEmbeddingsType
        if "SDXL" in str(type(pipe_img2img)):
            compel_proc = Compel(tokenizer=[pipe_img2img.tokenizer, pipe_img2img.tokenizer_2], text_encoder=[pipe_img2img.text_encoder, pipe_img2img.text_encoder_2], returned_embeddings_type=ReturnedEmbeddingsType.PENULTIMATE_HIDDEN_STATES_NON_NORMALIZED, requires_pooled=[False, True], truncate_long_prompts=False, device=pipe_img2img.device)
            cond, pooled = compel_proc(prompt_str)
            neg_cond, neg_pooled = compel_proc(req.negative_prompt or "")
            try:
                cond, neg_cond = compel_proc.pad_conditioning_tensors_to_same_length([cond, neg_cond])
            except AttributeError:
                import torch
                empty_cond, _ = compel_proc("")
                while cond.shape[1] > neg_cond.shape[1]:
                    neg_cond = torch.cat([neg_cond, empty_cond], dim=1)
                while neg_cond.shape[1] > cond.shape[1]:
                    cond = torch.cat([cond, empty_cond], dim=1)
            
            # Robustly extract pooled embeddings whether they are list, tuple, or single tensor
            def extract_pool(p):
                if hasattr(p, 'shape'): return p
                if isinstance(p, (list, tuple)):
                    valid = [x for x in p if x is not None and hasattr(x, 'shape')]
                    if valid: return valid[0]
                return None
                
            pooled_ext = extract_pool(pooled)
            neg_pooled_ext = extract_pool(neg_pooled)
            
            if pooled_ext is None or neg_pooled_ext is None:
                # Fallback to diffusers encode_prompt for pooled if compel didn't generate them
                _, _, p_emb, np_emb = pipe_img2img.encode_prompt(prompt=prompt_str, prompt_2=prompt_str, device=pipe_img2img.device, num_images_per_prompt=1, do_classifier_free_guidance=True, negative_prompt=req.negative_prompt or "", negative_prompt_2=req.negative_prompt or "")
                pooled_ext = p_emb if pooled_ext is None else pooled_ext
                neg_pooled_ext = np_emb if neg_pooled_ext is None else neg_pooled_ext
                
            kwargs = {"prompt_embeds": cond, "pooled_prompt_embeds": pooled_ext, "negative_prompt_embeds": neg_cond, "negative_pooled_prompt_embeds": neg_pooled_ext}
        else:
            compel_proc = Compel(tokenizer=pipe_img2img.tokenizer, text_encoder=pipe_img2img.text_encoder, truncate_long_prompts=False, device=pipe_img2img.device)
            cond = compel_proc(prompt_str)
            neg_cond = compel_proc(req.negative_prompt or "")
            try:
                cond, neg_cond = compel_proc.pad_conditioning_tensors_to_same_length([cond, neg_cond])
            except AttributeError:
                import torch
                empty_cond, _ = compel_proc("")
                while cond.shape[1] > neg_cond.shape[1]:
                    neg_cond = torch.cat([neg_cond, empty_cond], dim=1)
                while neg_cond.shape[1] > cond.shape[1]:
                    cond = torch.cat([cond, empty_cond], dim=1)
            kwargs = {"prompt_embeds": cond, "negative_prompt_embeds": neg_cond}

        try:
            with torch.inference_mode():
                image = pipe_img2img(**kwargs, image=init_image, strength=req.denoising_strength, num_inference_steps=req.steps, guidance_scale=req.cfg_scale, generator=generator).images[0]
        except Exception as pipe_err:
            pe_type = type(kwargs.get('pooled_prompt_embeds'))
            pe_val = kwargs.get('pooled_prompt_embeds') is None
            raise RuntimeError(f"PIPE CRASH! kwargs keys: {list(kwargs.keys())}. pooled type: {pe_type}. Is it None? {pe_val}. Original error: {pipe_err}")

    except Exception as e:
        import traceback
        import textwrap
        from PIL import Image, ImageDraw
        traceback.print_exc()
        err_msg = traceback.format_exc()
        print(f"Compel processing or generation failed: {e}. Returning error as image.")
        img = Image.new('RGB', (req.width, req.height), color = (30, 30, 30))
        d = ImageDraw.Draw(img)
        y_text = 20
        for line in err_msg.split('\n'):
            for wrap_line in textwrap.wrap(line, width=70):
                d.text((20, y_text), wrap_line, fill=(255, 100, 100))
                y_text += 20
        image = img

    if loaded_adapters:
        try: pipe_img2img.delete_adapters(loaded_adapters)
        except: pass

    return {
        "images": [_encode_image_to_base64(image)],
        "source": f"Google Colab Cloud GPU ({torch.cuda.get_device_name(0)})"
    }

@app.post("/sdapi/v1/img2img")
def img2img(req: Img2ImgRequest):
    task_id = str(uuid.uuid4())
    tasks[task_id] = {"status": "processing"}
    threading.Thread(target=_bg_runner, args=(task_id, _do_img2img, req)).start()
    return {"task_id": task_id}

def _do_interrogate(req: InterrogateRequest):
    _load_tagger()
    image = _decode_base64_image(req.image)
    target_size = 448
    img = image.convert("RGB")
    w, h = img.size
    max_dim = max(w, h)
    pad_img = Image.new("RGB", (max_dim, max_dim), (255, 255, 255))
    pad_img.paste(img, (max_dim//2 - w//2, max_dim//2 - h//2))
    img = pad_img.resize((target_size, target_size), Image.BICUBIC)
    image_array = np.array(img, dtype=np.float32)
    image_array = image_array[:, :, ::-1] # RGB to BGR
    image_array = np.expand_dims(image_array, axis=0)
    input_name = _tagger_session.get_inputs()[0].name
    output_name = _tagger_session.get_outputs()[0].name
    preds = _tagger_session.run([output_name], {input_name: image_array})[0][0]
    threshold = 0.35
    tag_preds = preds[4:]
    tag_names = _tagger_tags[4:]
    final_tags = []
    for p, name in zip(tag_preds, tag_names):
        if p > threshold:
            final_tags.append(name.replace('_', ' '))
    _unload_tagger()
    return {"caption": ", ".join(final_tags)}

@app.post("/sdapi/v1/interrogate")
def interrogate(req: InterrogateRequest):
    task_id = str(uuid.uuid4())
    tasks[task_id] = {"status": "processing"}
    threading.Thread(target=_bg_runner, args=(task_id, _do_interrogate, req)).start()
    return {"task_id": task_id}

def _do_upscale(req: UpscaleRequest):
    _load_upscaler()
    image = _decode_base64_image(req.image)
    img_bgr = np.array(image)[:, :, ::-1]
    output, _ = _upscaler.enhance(img_bgr, outscale=req.upscaling_resize)
    result_image = Image.fromarray(output[:, :, ::-1])
    _unload_upscaler()
    return {"images": [_encode_image_to_base64(result_image)], "source": "Real-ESRGAN"}

@app.post("/sdapi/v1/extra-single-image")
def upscale(req: UpscaleRequest):
    task_id = str(uuid.uuid4())
    tasks[task_id] = {"status": "processing"}
    threading.Thread(target=_bg_runner, args=(task_id, _do_upscale, req)).start()
    return {"task_id": task_id}

def _do_face_fix(req: FaceFixRequest):
    _load_face_restorer()
    image = _decode_base64_image(req.image)
    img_bgr = np.array(image)[:, :, ::-1]
    _, _, output = _face_restorer.enhance(img_bgr, has_aligned=False, only_center_face=False, paste_back=True)
    result_image = Image.fromarray(output[:, :, ::-1])
    _unload_face_restorer()
    return {"images": [_encode_image_to_base64(result_image)], "source": "GFPGAN"}

@app.post("/sdapi/v1/face-fix")
def face_fix(req: FaceFixRequest):
    task_id = str(uuid.uuid4())
    tasks[task_id] = {"status": "processing"}
    threading.Thread(target=_bg_runner, args=(task_id, _do_face_fix, req)).start()
    return {"task_id": task_id}

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"), daemon=True).start()
time.sleep(2)
tunnel = try_cloudflare(port=8000)
print(f"\n🎉 COPY THIS URL: {tunnel.tunnel}\n")

















